# N1 · online softmax (flash attention 的核心思想)

> 复用 `src/capstone_softmax.py` · 对比 naive softmax (两遍/存全部) vs online softmax (一遍/常数内存)。
> online softmax 是 flash attention 不爆显存的关键 (接本专题讲义)。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import capstone_softmax as cs
print('cuda-essentials src 就绪 (纯 Python 模拟 kernel)')

## 1. naive vs online softmax: 结果一致, 但 online 单遍 + 数值稳定

In [ ]:
x = [3.0, 1.0, 0.2, 5.0, 2.5, -1.0, 4.0]
naive = cs.softmax_naive(x)
online = cs.softmax_online(x)
print('naive  softmax:', [round(v,4) for v in naive])
print('online softmax:', [round(v,4) for v in online])
print('两者最大差异:', round(max(abs(a-b) for a,b in zip(naive, online)), 8))
print('→ 结果一致; online 只过一遍数据 + 在线维护 max/sum (常数内存, 数值稳定)。')

## 2. 为什么 online softmax 是 flash attention 的钥匙

In [ ]:
print('''online softmax 机制 (本专题讲义):
  naive:  ① 求 max ② 求 sum(exp(x-max)) ③ 除  -> 要存全部 x (两/三遍)
  online: 边读边更新 running_max 和 running_sum, 遇到更大的 max 就 rescale
          -> 单遍, 常数内存, 不存全部分数

为什么关键 (flash attention):
  attention 的 softmax 在 S×S 的分数矩阵上 -> 存全部 = O(S^2) 显存爆炸
  online softmax 让 softmax 能分块 (tiling) 流式算 -> flash attention 不存全矩阵
  = kernel-engineering 专题 flash attention 的数学基础''')

> 本专题其余 src (`vector_add` / `gemm_tiled` / `reduce_kernel` / `shared_memory` / `coalescing` / `warp_primitives`) 是 CUDA 核心概念的纯 Python 模拟, 可在对应讲义里运行其 `_self_test()`。